Step 12: Deploy the Created Model in a Cloud Environment with Hugging Face Spaces

In [4]:
"""
This step will require manual actions and terminal commands.

Preparation:
    Create a Hugging Face Account: Sign up at huggingface.co.

    Install Git LFS: You might need to install Git LFS (Large File Storage) to manage large files (like model files).
    Install Hugging Face CLI:
    Login: Log in to your Hugging Face account in the terminal.

Create a Git Repository:
    Create a New Directory and Navigate into It: This will be the main repository you upload to your Hugging Face Space.
    Copy the Model Folder: Copy the model folder we created in previous steps into this new directory.
"""

'\nThis step will require manual actions and terminal commands.\n\nPreparation:\n    Create a Hugging Face Account: Sign up at huggingface.co.\n\n    Install Git LFS: You might need to install Git LFS (Large File Storage) to manage large files (like model files).\n    Install Hugging Face CLI:\n    Login: Log in to your Hugging Face account in the terminal.\n\nCreate a Git Repository:\n    Create a New Directory and Navigate into It: This will be the main repository you upload to your Hugging Face Space.\n    Copy the Model Folder: Copy the model folder we created in previous steps into this new directory.\n'

Create the Streamlit Application (app.py)

In [7]:
%%writefile sarcopenia-predictor-space/app.py
import streamlit as st
import pandas as pd
import requests
import json
import xgboost as xgb
import numpy as np
import joblib
import os

from huggingface_hub import hf_hub_download

# Hugging Face model repository details
REPO_ID = "carpediem61/sarco_model"
FILENAME = "xgb_model.json"
DATA_TYPES_FILE = "data_types.csv" # CSV file for feature data types

@st.cache_resource
def load_model():
    """
    Loads the XGBoost model from Hugging Face Hub.
    Uses st.cache_resource to cache the model for performance.
    """
    try:
        model_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
        booster = xgb.Booster()
        booster.load_model(model_path)
        return booster
    except Exception as e:
        st.error(f"An error occurred while loading the model: {e}")
        st.stop() # Stop the app if model cannot be loaded

booster = load_model()

st.set_page_config(layout="wide")

st.title("Sarcopenia Risk Prediction Application 🩺")
st.markdown("""
This application uses a machine learning model to predict sarcopenia risk based on the data you enter.
""")

st.sidebar.header("Input Features")

# Function to load feature metadata from data_types.csv
def load_feature_metadata(file_name):
    """
    Loads feature columns and identifies boolean/int64 features from a CSV file.
    Expected CSV columns: 'Column Name', 'Data Type'.
    """
    feature_columns = []
    boolean_features = []
    
    # Construct the full path to the data_types.csv file
    # Assuming data_types.csv is in the 'model' directory
    file_path = os.path.join("model", file_name)

    if os.path.exists(file_path):
        try:
            df_data_types = pd.read_csv(file_path, encoding='utf-8')

            # --- MODIFICATION HERE: Changed column names to English ---
            if 'Column Name' in df_data_types.columns and 'Data Type' in df_data_types.columns:
                for index, row in df_data_types.iterrows():
                    col_name = row['Column Name'] # Changed from 'Sutun Adı'
                    data_type = str(row['Data Type']).strip().lower() # Changed from 'Veri Tipi'

                    feature_columns.append(col_name)

                    # Identify boolean or integer features that should use radio buttons (0 or 1)
                    # We check for 'bool', 'int64', and 'int', as these are common representations
                    # of 0/1 values. 'Int64' is Pandas' nullable integer type.
                    if data_type in ['bool', 'int64', 'int', 'int32', 'int16', 'int8', 'uint64', 'uint32', 'uint16', 'uint8', 'int64dtytpe']:
                        boolean_features.append(col_name)
            else:
                st.error(f"Columns 'Column Name' or 'Data Type' not found in '{file_path}'.")
                st.info("Please ensure your CSV file has the correct column headers: 'Column Name' and 'Data Type'.")
        except Exception as e:
            st.error(f"An error occurred while reading '{file_path}': {e}")
            st.info("Please ensure the file format is correct (CSV).")
    else:
        st.error(f"'{file_path}' not found. Please ensure the 'model' folder is copied correctly and contains '{DATA_TYPES_FILE}'.")
        st.info("To create an example 'data_types.csv' file, please follow the previous steps.")
        st.stop() # Stop the app if file is not found

    return feature_columns, boolean_features

# Load FEATURE_COLUMNS and BOOLEAN_FEATURES from the CSV file
FEATURE_COLUMNS, BOOLEAN_FEATURES = load_feature_metadata(DATA_TYPES_FILE)

# If FEATURE_COLUMNS is empty, it means there was an error loading the metadata,
# and the app would have already stopped via st.stop() in load_feature_metadata.

input_data = {}
for col in FEATURE_COLUMNS:
    if col in BOOLEAN_FEATURES:
        # For boolean or integer (0/1) features, use a radio button
        input_data[col] = st.sidebar.radio(
            f"{col} (0: No, 1: Yes)", # More descriptive label
            options=[0, 1],
            index=0, # Default to 0
            key=f"radio_{col}" # Unique key for each widget
        )
    else:
        # For other numerical features, use a number input
        input_data[col] = st.sidebar.number_input(f"Enter value for: {col}", value=0.0, key=f"number_{col}")

# Make Prediction button
if st.sidebar.button("Make Prediction"):
    try:
        # Ensure model inference functions are available
        # These are expected to be in 'model/inference.py'
        from model.inference import model_fn, input_fn, predict_fn, output_fn

        # Load model into session_state if not already loaded
        if 'loaded_model' not in st.session_state:
            st.session_state.loaded_model = model_fn("model/")

        # Prepare input data for prediction
        processed_input_df = pd.DataFrame([input_data])
        
        # Ensure all FEATURE_COLUMNS are present in the input DataFrame
        # and in the correct order. Fill missing columns with 0.0 as a fallback.
        # This is crucial for the model's input consistency.
        for col in FEATURE_COLUMNS:
            if col not in processed_input_df.columns:
                processed_input_df[col] = 0.0
        processed_input_df = processed_input_df[FEATURE_COLUMNS] # Ensure correct order

        # Make prediction
        prediction = predict_fn(processed_input_df, st.session_state.loaded_model)

        st.subheader("Prediction Result:")
        if prediction[0] == 1:
            st.error("🚨 High Sarcopenia Risk!")
        else:
            st.success("✅ Low Sarcopenia Risk.")

        st.write(f"Model Prediction (0: Low Risk, 1: High Risk): **{prediction[0]}**")

    except Exception as e:
        st.error(f"An error occurred while making the prediction: {e}")
        st.info("Please ensure all dependencies are installed and files in the 'model' folder (including model/data_types.csv) are correct.")

st.sidebar.markdown("---")
st.sidebar.info("You can make predictions by entering values in the boxes on the left.")

st.markdown("### Model Features and Data Information")
# Display the input data DataFrame for user review
st.write(pd.DataFrame([input_data]))

Overwriting sarcopenia-predictor-space/app.py


Create README.md: A description file for your Hugging Face Space page.

In [5]:
%%writefile sarcopenia-predictor-space/README.md
---
title: Sarcopenia Risk Predictor
emoji: 🩺
colorFrom: green
colorTo: blue
sdk: streamlit
python_version: 3.9
app_file: app.py
sdk_version: 1.46.1
---

Sarcopenia Risk Predictor

This is a Streamlit application that predicts the risk of sarcopenia based on various input features using an XGBoost model.

How to Use

1.  Enter the values for each feature in the sidebar.
2.  Click the "Make Prediction" button.
3.  The prediction (High Risk or Low Risk) will be displayed.

Model Details

The model is an XGBoost Classifier trained on the `sarcopenia-data-set_columns_corrected_13_optimized.csv` dataset.

---
title: Sarcopenia Predictor
emoji: 🚀
colorFrom: red
colorTo: red
sdk: docker
app_port: 8501
tags:
- streamlit
pinned: false
short_description: used for sarcopenia prediction
license: mit
---

Overwriting sarcopenia-predictor-space/README.md


Initialize and Push Git Repository: Now we will upload these files to Hugging Face Spaces
(This step is performed over cmd terminal)

In [ ]:
#completed

Overwriting sarcopenia-predictor-space/app.py


Step 13: Test the Deployed Model with Test Data

In [ ]:
#completed